# Intro til Machine Learning — 2: PyTorch & gradient descent ⚡

I regressionsemnet fandt I den bedste rette linje gennem en punktsky **med en formel**.
Det var smart — men formlen findes kun, fordi lineær regression er et simpelt problem.
For neurale netværk findes der *ingen formel*. I stedet skal computeren **lede sig frem**
til de bedste tal, lidt ad gangen. Den metode hedder **gradient descent**, og den er
motoren i stort set al moderne AI.

I denne notebook møder I:
1. **Tensorer** — PyTorch' svar på numpy-arrays (afsnit 5)
2. **Autograd** — PyTorch' indbyggede, automatiske differentiering (afsnit 6)
3. **Gradient descent** — og lineær regression én gang til, nu på den nye måde (afsnit 7)

> Husk: der er flere opgaver end I kan nå. ⭐ = til de hurtige, 🐛 = find den bevidste fejl.

## Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
# Henter vores lille plottehjælper fra GitHub (kun matplotlib-pynt, ingen magi)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/hjaelpefunktioner.py

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from hjaelpefunktioner import plot_gd_skridt

torch.manual_seed(42)
np.random.seed(42)

> 🚨 **Plan B:** Fejler `wget`-cellen, så hent `hjaelpefunktioner.py` fra GitHub-repoet
> (mappen `Intro-ML`) og upload den manuelt via Colabs filpanel (📁-ikonet til venstre).

Og så skal vi selvfølgelig have Pokémonerne med igen:

In [ ]:
sti = kagglehub.dataset_download("abcsds/pokemon")
df = pd.read_csv(sti + "/Pokemon.csv")

# 🚨 Plan B hvis Kaggle driller — fjern #'et:
# df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/Pokemon.csv")

df.head(3)

# 5: Tensorer — numpy med superkræfter

PyTorch' grundbyggesten hedder en **tensor**. Den ligner et numpy-array til forveksling —
samme indeksering, samme regneoperationer, samme `.shape` — men den har to superkræfter,
som numpy ikke har:

1. Den kan **huske, hvordan den blev udregnet**, og selv finde gradienter (afsnit 6!).
2. Den kan bo på et grafikkort og regne vanvittigt hurtigt (det behøver vi ikke i dag).

Lad os prøve:

In [ ]:
t = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(t)
print("shape:", t.shape)
print("dtype:", t.dtype)

In [ ]:
# Regneoperationer virker præcis som i numpy — på alle tal på én gang
print(t + 10)
print(t * 2)
print(t ** 2)
print("gennemsnit:", t.mean())

In [ ]:
# Indeksering og slicing — også som numpy
print(t[0])
print(t[1:3])

## Fra pandas og numpy til tensor (og tilbage)

Vejen fra jeres DataFrame til en tensor går via `.values` (som I kender) og `torch.tensor`.
Vi skriver `dtype=torch.float32` — kommatal i 32 bit er dét format, stort set al ML bruger:

In [ ]:
hp = torch.tensor(df["HP"].values, dtype=torch.float32)
print(hp[:8])
print("shape:", hp.shape)

tilbage = hp.numpy()   # ...og tilbage til numpy, hvis man vil
print(type(tilbage))

## Matrix-multiplikation og reshape

To ting, vi får hårdt brug for, når netværkene kommer:

- `@` ganger matricer sammen. Husk reglen: **(n, m) @ (m, k) → (n, k)** — de indre
  dimensioner skal matche, ellers protesterer PyTorch.
- `reshape` omformer en tensor til en anden facon — så længe antallet af tal passer.

In [ ]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])   # shape (3, 2)
B = torch.tensor([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0]])       # shape (2, 3)
print((A @ B).shape)                                        # (3, 2) @ (2, 3) → (3, 3)

In [ ]:
tal = torch.arange(12.0)     # 0, 1, ..., 11
print(tal.reshape(3, 4))
print(tal.reshape(4, 3).shape)

### Opgaver

##### Opgave 5.1
**Forudsig outputtet, FØR I kører cellen** — skriv jeres gæt som en kommentar. Kør den så,
og tjek jer selv. Ændr derefter tallene og gæt igen, indtil I rammer rigtigt to gange i træk.

In [ ]:
t = torch.tensor([2.0, 4.0, 6.0])
# (t / 2 + 1) → [2., 3., 4.]  (hvert tal halveres og får 1 lagt til)
print(t / 2 + 1)
# t[-1] - t[0] → 4.  (sidste minus første: 6 - 2)
print(t[-1] - t[0])
# (t ** 2).sum() → 56.  (4 + 16 + 36)
print((t ** 2).sum())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Pointen: alle regneoperationer rammer ALLE tal i tensoren på én gang — præcis som i numpy.</span>

##### Opgave 5.2
Konvertér Pokémon-kolonnen `Attack` til en tensor — udfyld de to huller. Print derefter
dens `shape` og dens største værdi med `.max()`.

In [ ]:
attack = torch.tensor(df["Attack"].values, dtype=torch.float32)
print(attack.shape)
print(attack.max())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Shape er <code>torch.Size([800])</code> og max er 190 (MewtwoMega Mewtwo X). Mønstret <code>torch.tensor(df["kolonne"].values, dtype=torch.float32)</code> bruger vi resten af emnet.</span>

##### Opgave 5.3 🐛
Cellen forsøger at gange to matricer, men PyTorch protesterer. Kør den, læs fejlbeskeden
(den fortæller præcis, hvilke shapes der ikke passer sammen), og ret `B`, så
multiplikationen kan lade sig gøre. Hvad bliver resultatets shape?

In [ ]:
A = torch.ones(3, 2)
B = torch.ones(2, 3)   # de INDRE dimensioner skal matche: (3, 2) @ (2, 3)
print(A @ B)
print("resultatets shape:", (A @ B).shape)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> (3, 2) @ (3, 2) er umuligt — den indre dimension (2 og 3) matcher ikke. Med B som (2, 3) bliver resultatet (3, 3). Man kunne også have transponeret: <code>A @ B.T</code>. Denne fejlbesked bliver jeres trofaste følgesvend i resten af jeres ML-liv. 🙃</span>

##### Opgave 5.4
Udfyld cellen, så den beregner gennemsnit og spredning af `Speed`-kolonnen som tensor.
Sammenlign med tallene fra `df.describe()` i notebook 1 — passer de?

In [ ]:
speed = torch.tensor(df["Speed"].values, dtype=torch.float32)
print("gennemsnit:", speed.mean())
print("spredning: ", speed.std())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Gennemsnit ≈ 68,3 og spredning ≈ 29,1 — samme tal som pandas' <code>describe()</code> gav (torch bruger samme formel for std som pandas).</span>

##### Opgave 5.5
En tensor med 12 tal kan omformes på mange måder. Prøv `reshape(3, 4)`, `reshape(4, 3)`
og `reshape(2, 2, 3)` — og til sidst `reshape(5, 3)`. Hvad sker der i det sidste tilfælde,
og hvorfor?

In [ ]:
tal = torch.arange(12.0)
print(tal.reshape(3, 4))
print(tal.reshape(4, 3))
print(tal.reshape(2, 2, 3))   # tensorer kan sagtens have 3 (eller flere) dimensioner!
# tal.reshape(5, 3) fejler: 5 * 3 = 15 pladser, men vi har kun 12 tal

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>reshape</code> kan kun omforme, ikke tilføje/fjerne tal: 3·4 = 4·3 = 2·2·3 = 12 ✓, men 5·3 = 15 ✗ → RuntimeError. Læg mærke til at (2, 2, 3) giver en 3-dimensionel tensor — i notebook 5 skal I se billeder som netop sådan nogle.</span>

##### Opgave 5.6 ⭐
Standardisér **alle seks kamp-stats på én gang**: cellen bygger en (800, 6)-tensor, og I skal
genbruge formlen fra opgave 4.7 — men nu skal `mean` og `std` regnes **pr. kolonne**, hvilket
gøres med `dim=0` (dimension 0 = ned langs rækkerne). Tjek at hver kolonnes gennemsnit ender
tæt på 0.

In [ ]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X = torch.tensor(df[stats].values, dtype=torch.float32)
print("X shape:", X.shape)

X_std = (X - X.mean(dim=0)) / X.std(dim=0)
print(X_std.mean(dim=0))   # seks tal, alle ~0

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>dim=0</code> betyder "regn ned langs rækkerne", så man får ét gennemsnit pr. kolonne (6 tal) i stedet for ét samlet. Uden <code>dim</code> ville alle seks stats blive skaleret med SAMME gennemsnit — og så taler kolonnerne ikke længere samme sprog. Denne (800, 6)-tensor er præcis, hvad vi fodrer netværket med i notebook 3!</span>

##### Opgave 5.7
Tensorer ligner numpy-arrays til forveksling — indtil videre har de bare været "numpy med
andet navn". Hvad tror I, tensorer kan, som numpy-arrays ikke kan? (Svaret afsløres i næste
afsnit... men gæt først!)

*Skriv jeres gæt her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Den store forskel: en tensor kan huske hele sin egen udregningshistorie og automatisk regne gradienter baglæns igennem den (autograd — næste afsnit!). Og så kan tensorer bo på grafikkort, hvilket gør store modeller tusindvis af gange hurtigere at træne.</span>

# 6: Autograd — den automatiske hældningsmåler

Nu til tensorernes superkraft nummer ét. I kender **differentialkvotienten** fra gymnasiet:
$f'(x)$ måler, hvor stejl grafen for $f$ er i punktet $x$ — og hvilken vej den hælder.

PyTorch kan regne den ud **automatisk**, uanset hvor indviklet funktionen er. Det virker
sådan her:

1. Opret en tensor med `requires_grad=True` — "hold øje med denne variabel!"
2. Regn noget ud med den — PyTorch husker i smug alle mellemregninger.
3. Kald `.backward()` på resultatet — PyTorch regner baglæns og finder gradienten.
4. Aflæs gradienten i `.grad`.

Lad os tjekke, at PyTorch kan differentiere $f(x) = x^2$ (vi ved jo, at $f'(x) = 2x$):

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print("PyTorch siger: f'(3) =", x.grad)
print("Og i hånden:   f'(3) = 2 · 3 =", 2 * 3)

PyTorch fandt $f'(3) = 6$ — uden at kende formlen $f'(x) = 2x$! Den har simpelthen
regnet baglæns gennem udregningen. Ordet **gradient** er bare ML-sprog for
differentialkvotient (og den generaliserer til funktioner af mange variable).

Lad os *se* hvad gradienten betyder — den er hældningen af tangenten i punktet:

In [ ]:
xs = np.linspace(-4, 4, 100)
plt.plot(xs, xs ** 2, label="f(x) = x²")

# tangenten i x = 3 har hældningen f'(3) = 6:
haeldning = x.grad.item()
plt.plot(xs, haeldning * (xs - 3) + 9, "--", color="crimson",
         label=f"tangent i x=3 (hældning {haeldning:.0f})")
plt.scatter([3], [9], color="crimson", zorder=3)
plt.ylim(-5, 20)
plt.legend()
plt.title("Gradienten er tangentens hældning")
plt.show()

## Hvorfor er det smart?

Gradientens **fortegn** fortæller, hvilken vej grafen går *op* ad bakke. Positiv gradient i
et punkt? Så bliver $f$ større, hvis $x$ vokser. I machine learning har vi en **tabsfunktion**
(husk MSE fra regressionsemnet — den måler, hvor *forkert* modellen er), og vi vil gøre tabet
så **lille** som muligt. Så vi skal altid gå **imod** gradienten — *ned* ad bakken. 🏔️⬇️

Det er hele hemmeligheden bag næste afsnit. Men først: prøv selv autograd.

### Opgaver

##### Opgave 6.1
Koden finder gradienten af $f(x) = x^2$ i $x = 3$. Ændr den, så den i stedet finder
gradienten af $f(x) = x^3 + 2x$ i $x = 2$. Regn først facit i hånden
($f'(x) = 3x^2 + 2$) — og tjek så, om PyTorch er enig.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3 + 2 * x
y.backward()
print(x.grad)   # f'(2) = 3 · 2² + 2 = 14 ✓

<span style="color:red"><b>LØSNINGSFORSLAG:</b> f'(x) = 3x² + 2, så f'(2) = 14 — og det er præcis, hvad PyTorch svarer. Automatisk differentiering er ikke en tilnærmelse; den er matematisk eksakt.</span>

##### Opgave 6.2
Udfyld de to manglende linjer, så gradienten af $f(x) = 5x^2 - x$ i $x = 1$ bliver beregnet
og printet.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = 5 * x ** 2 - x
y.backward()
print(x.grad)   # f'(x) = 10x - 1, så f'(1) = 9

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>.backward()</code> sætter regnemaskinen i gang, og resultatet lander i <code>x.grad</code>. Facit: 9.</span>

##### Opgave 6.3 🐛
Nogen ville finde en gradient, men `x.grad` er `None`, og `.backward()` brokker sig oven
i købet. Der mangler noget helt afgørende ved oprettelsen af tensoren — find det og ret det.

In [ ]:
x = torch.tensor(4.0, requires_grad=True)   # ← det var dét, der manglede
y = x ** 2
y.backward()
print(x.grad)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Uden <code>requires_grad=True</code> holder PyTorch ikke øje med tensoren — der bliver ikke husket nogen mellemregninger, og så er der ingenting at regne baglæns igennem. Det er slået FRA som standard, fordi bogholderiet koster tid og hukommelse.</span>

##### Opgave 6.4
Kør cellen nedenfor. Gradienten af $x^2$ i $x=3$ er 6... men der bliver printet **12**!
Kig på loopet: `.backward()` bliver kaldt to gange, uden at `.grad` bliver nulstillet
imellem. Hvad gør PyTorch åbenbart med gradienter, når man kalder `.backward()` flere gange?

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
for i in range(2):
    y = x ** 2
    y.backward()
print(x.grad)   # 12?! Ikke 6?

<span style="color:red"><b>LØSNINGSFORSLAG:</b> PyTorch <b>lægger gradienter oveni</b> (akkumulerer) i stedet for at overskrive: 6 + 6 = 12. Derfor SKAL man nulstille gradienterne mellem hvert skridt, når man træner — ellers blander gamle gradienter sig med de nye. Husk denne opgave, når I møder <code>zero_grad()</code> om lidt — og når loop 7.5 opfører sig mystisk. 😈</span>

##### Opgave 6.5 ⭐
Autograd virker også med **flere variable**. Find gradienten af $f(a, b) = a^2 + 3b$ i
punktet $(a, b) = (2, 5)$ — udfyld hullerne. I hånden: $\frac{\partial f}{\partial a} = 2a$
og $\frac{\partial f}{\partial b} = 3$. Hvad fortæller de to tal hver især?

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(5.0, requires_grad=True)
f = a ** 2 + 3 * b
f.backward()
print("gradient for a:", a.grad)   # 2a = 4
print("gradient for b:", b.grad)   # 3

<span style="color:red"><b>LØSNINGSFORSLAG:</b> a.grad = 4 betyder: "gør du a lidt større, vokser f cirka 4× så hurtigt". b.grad = 3 tilsvarende for b. ÉT kald til <code>.backward()</code> gav gradienten for BEGGE variable på én gang — et neuralt netværk har millioner af variable, og autograd klarer dem alle sammen i ét hug. Det er derfor, PyTorch findes.</span>

##### Opgave 6.6
Hvis gradienten i et punkt er **negativ** — skal vi så gøre $x$ større eller mindre for at
komme *ned* ad bakken? Og hvad hvis den er positiv? Formulér en generel regel med jeres
egne ord.

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Negativ gradient = grafen falder mod højre = gå til HØJRE (gør x større) for at komme ned. Positiv gradient = gå til venstre. Den generelle regel: <b>flyt dig altid i modsat retning af gradienten</b> — altså <code>x_ny = x - skridtlængde · gradient</code>. Det er bogstaveligt talt formlen for gradient descent, som I nu er klar til!</span>

# 7: Gradient descent — at rulle ned ad bakken 🏔️

Nu samler vi det hele. Opskriften på **gradient descent** er tre linjer:

1. Stå et sted (start med en tilfældig værdi af $a$).
2. Mærk hældningen (regn gradienten).
3. Tag et lille skridt **ned** ad bakken: $a_{\text{ny}} = a - \text{læringsrate} \cdot \text{gradient}$.

...og gentag med en for-løkke, indtil man står i bunden. **Læringsraten** bestemmer, hvor
store skridt man tager — det bliver vigtigt om lidt.

Vi starter med en funktion, hvor vi kan se facit med det blotte øje: $f(a) = (a-3)^2$ har
selvfølgelig sit minimum i $a = 3$. Differentialkvotienten kan I selv regne: $f'(a) = 2(a-3)$.

In [ ]:
def f(a):
    return (a - 3) ** 2

def f_maerke(a):              # f'(a) = 2(a - 3), regnet i hånden
    return 2 * (a - 3)

laeringsrate = 0.1
a = -2.0                      # startgæt — langt fra minimum
skridt = [a]

for i in range(20):
    a = a - laeringsrate * f_maerke(a)    # ← SELVE gradient descent-linjen
    skridt.append(a)

print("Efter 20 skridt står vi i a =", round(a, 4))
plot_gd_skridt(f, skridt, titel="Gradient descent på f(a) = (a-3)²")

Sådan! Fra startgættet $a = -2$ ruller vi ned og lander (næsten) i $a = 3$. Bemærk at
skridtene bliver mindre og mindre af sig selv — tæt på bunden er gradienten lille, så
$\text{læringsrate} \cdot \text{gradient}$ bliver også lille. Elegant, ikke?

## Læringsraten: for lille, tilpas, for stor

Læringsraten er gradient descents vigtigste indstilling. Se, hvad der sker med en
tilpas (0.1) og en alt for stor (1.05) læringsrate:

In [ ]:
fig, akser = plt.subplots(1, 2, figsize=(12, 4))

for akse, lr in zip(akser, [0.1, 1.05]):
    a = -2.0
    skridt = [a]
    for i in range(15):
        a = a - lr * f_maerke(a)
        skridt.append(a)
    xs = np.linspace(min(skridt) - 1, max(skridt) + 1, 200)
    akse.plot(xs, f(xs), color="steelblue")
    akse.plot(skridt, [f(s) for s in skridt], "o-", color="crimson", alpha=0.7)
    akse.set_title(f"læringsrate = {lr} → ender i a = {a:.1f}")
plt.show()

Med læringsrate 1.05 tager vi så store skridt, at vi **hopper forbi** bunden og lander
højere oppe på den anden side — igen og igen. Tabet *vokser* for hvert skridt. 💥

## Lineær regression — nu uden formel

Tilbage til det, I kender: find den bedste linje $y = ax + b$ gennem en punktsky. I
regressionsemnet brugte I en formel. Nu lader vi gradient descent finde $a$ og $b$ — og
denne gang må autograd regne gradienterne (ingen håndkraft!).

Vi prøver at forudsige en Pokémons `Total` ud fra dens `Attack`. Begge kolonner
standardiseres først (notebook 1-tricket!), så tallene er pæne:

In [ ]:
x = torch.tensor(df["Attack"].values, dtype=torch.float32)
y = torch.tensor(df["Total"].values, dtype=torch.float32)

x = (x - x.mean()) / x.std()
y = (y - y.mean()) / y.std()

plt.scatter(x, y, alpha=0.3)
plt.xlabel("Attack (standardiseret)")
plt.ylabel("Total (standardiseret)")
plt.show()

In [ ]:
a = torch.tensor(0.0, requires_grad=True)   # start: en flad linje
b = torch.tensor(0.0, requires_grad=True)
laeringsrate = 0.1
tab_historik = []

for epoke in range(100):
    y_hat = a * x + b                      # 1. modellens gæt for ALLE 800 Pokémon
    tab = ((y_hat - y) ** 2).mean()        # 2. MSE: hvor forkerte er vi i snit?
    tab.backward()                         # 3. autograd finder gradienterne
    with torch.no_grad():                  # 4. tag ét skridt ned ad bakken
        a -= laeringsrate * a.grad
        b -= laeringsrate * b.grad
    a.grad.zero_()                         # 5. nulstil gradienter (husk opgave 6.4!)
    b.grad.zero_()
    tab_historik.append(tab.item())

print(f"Linjen blev: y = {a.item():.3f}·x + {b.item():.3f}")

(Detalje: `with torch.no_grad():` betyder "dette er ikke en del af modellen, kig væk,
autograd" — selve opdaterings-skridtet skal jo ikke differentieres. Og `.zero_()` nulstiller
gradienterne, så de ikke hober sig op, jf. opgave 6.4.)

Én gennemløbning af alle data kaldes en **epoke** — deraf løkkevariablen. Lad os se
resultatet og **tabskurven** — ML-folkets vigtigste diagnoseværktøj. Den skal falde og
flade ud:

In [ ]:
fig, akser = plt.subplots(1, 2, figsize=(12, 4))

akser[0].scatter(x, y, alpha=0.3)
xs = torch.linspace(x.min(), x.max(), 100)
with torch.no_grad():
    akser[0].plot(xs, a * xs + b, color="crimson", linewidth=2)
akser[0].set_title("Den fundne linje")
akser[0].set_xlabel("Attack (std)")
akser[0].set_ylabel("Total (std)")

akser[1].plot(tab_historik)
akser[1].set_title("Tabskurven — skal falde og flade ud")
akser[1].set_xlabel("epoke")
akser[1].set_ylabel("MSE-tab")
plt.show()

## Flere features på én gang

Hvorfor nøjes med `Attack`? Med matrix-multiplikation kan linjen få **seks** hældninger —
én pr. stat: $\hat{y} = X\,\vec{w} + b$. Og her har Pokémon-datasættet en indbygget
facitliste: `Total` er jo bogstaveligt talt **summen** af de seks stats! Så den perfekte
model har alle seks vægte = 1 og bias = 0.

Denne gang standardiserer vi ikke, men deler alt med 100 (samme skalering for alle
kolonner, så facit stadig passer — og tallene stadig er pæne):

In [ ]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X = torch.tensor(df[stats].values, dtype=torch.float32) / 100
y = torch.tensor(df["Total"].values, dtype=torch.float32) / 100

w = torch.zeros(6, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
laeringsrate = 0.1

for epoke in range(3000):
    y_hat = X @ w + b
    tab = ((y_hat - y) ** 2).mean()
    tab.backward()
    with torch.no_grad():
        w -= laeringsrate * w.grad
        b -= laeringsrate * b.grad
    w.grad.zero_()
    b.grad.zero_()

print("vægte:", w.detach().numpy().round(3), "— facit: 1, 1, 1, 1, 1, 1")
print("bias:  ", round(b.item(), 4), "— facit: 0")
print("sluttab:", round(tab.item(), 6))

Gradient descent **opdagede selv**, at Total er summen af de seks stats — vi har på
intet tidspunkt fortalt den det! 🤯 Det er machine learning i en nøddeskal: mønstret lå i
dataene, og optimeringen fandt det.

## De indbyggede klodser: tabsfunktion og optimizer

Vores håndskrevne loop virker, men PyTorch har færdige klodser til delene: `nn.MSELoss()`
er vores MSE-linje, og `torch.optim.SGD` holder styr på opdatering + nulstilling. Samme
loop, nu med klodserne — læg mærke til **tre-trins-rytmen** i loopet, for den skal I kunne
i søvne: **`zero_grad()` → `backward()` → `step()`**

In [ ]:
import torch.nn as nn

model = nn.Linear(6, 1)                                  # 6 features ind → 1 tal ud (w OG b indbygget!)
tabsfunktion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoke in range(3000):
    optimizer.zero_grad()                    # 1. nulstil gamle gradienter
    y_hat = model(X).squeeze()               # (forward: modellens gæt)
    tab = tabsfunktion(y_hat, y)             # (regn tabet)
    tab.backward()                           # 2. find gradienterne
    optimizer.step()                         # 3. tag ét skridt ned ad bakken

print("vægte:", model.weight.detach().numpy().round(3))
print("bias: ", round(model.bias.item(), 4))

Samme resultat — bare med mindre bogholderi. (`.squeeze()` fjerner en overflødig
dimension: `model(X)` giver shape (800, 1), men `y` har shape (800).)

`nn.Linear(6, 1)` er i øvrigt jeres allerførste **netværkslag** — i notebook 3 stabler vi
sådan nogle oven på hinanden, og så har I et neuralt netværk. Men først: opgaver!

### Opgaver

##### Opgave 7.1
Kør gradient descent på $f(a) = (a-3)^2$ med læringsraterne **0.01, 0.1, 0.9 og 1.05**
(én ad gangen). Beskriv med jeres egne ord, hvad der sker i hvert af de fire tilfælde.

In [ ]:
for laeringsrate in [0.01, 0.1, 0.9, 1.05]:
    a = -2.0
    skridt = [a]
    for i in range(20):
        a = a - laeringsrate * f_maerke(a)
        skridt.append(a)
    print(f"lr = {laeringsrate}: ender i a = {round(a, 3)}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> 0.01: bittesmå skridt — efter 20 skridt er vi stadig langt fra 3 (for langsomt). 0.1: pæn rolig nedstigning, lander ~3. 0.9: hopper vildt frem og tilbage hen over bunden, men zigzagger sig alligevel ind mod 3. 1.05: hvert hop lander HØJERE oppe end før — det eksploderer (ender i a ≈ −31 og på vej væk). Læringsraten er altså en balance mellem tålmodighed og eksplosion.</span>

##### Opgave 7.2
Med læringsrate 0.1: prøv forskellige startpunkter (`a = -2.0`, `a = 100.0`, `a = 3.0`) og
forskellige antal skridt. Cirka hvor mange skridt kræver det at komme tættere på minimum
end 0.001, når man starter i $a = 100$? (Ret i tallene og aflæs!)

In [ ]:
laeringsrate = 0.1
a = 100.0
antal_skridt = 52   # ca. dér krydser afstanden 0.001
for i in range(antal_skridt):
    a = a - laeringsrate * f_maerke(a)
print("afstand til minimum:", abs(a - 3))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Hvert skridt ganger afstanden til minimum med 0,8 (fordi a_ny − 3 = 0,8·(a − 3) når lr = 0,1). Fra afstand 97 skal man bruge ca. 52 skridt, før 97 · 0,8ⁿ &lt; 0,001. Starter man i a = 3.0, står man allerede i minimum — gradienten er 0, og der sker ingenting.</span>

##### Opgave 7.3
Udfyld selve gradient descent-linjen (opdateringen af `a`) i skabelonen — resten står klar.

In [ ]:
laeringsrate = 0.1
a = -2.0
for i in range(30):
    a = a - laeringsrate * f_maerke(a)
print("ender i a =", round(a, 4))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>a = a - laeringsrate * f_maerke(a)</code> — minus, fordi vi går IMOD gradienten (jf. opgave 6.6).</span>

##### Opgave 7.4 🐛
Loopet nedenfor skulle finde minimum... men `a` stikker af i den helt forkerte retning og
bliver bare større og større! Find fortegnsfejlen og ret den.

In [ ]:
laeringsrate = 0.1
a = -2.0
skridt = [a]
for i in range(15):
    a = a - laeringsrate * f_maerke(a)   # MINUS: vi skal MOD gradienten, ned ad bakken
    skridt.append(a)
print("ender i a =", round(a, 1))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Der stod <code>+</code> i stedet for <code>-</code>: med plus følger man gradienten OP ad bakken — gradient <i>ascent</i> — og tabet vokser for hvert skridt. En klassisk fejl, som heldigvis er nem at få øje på: tabskurven stiger.</span>

##### Opgave 7.5 🐛
Dette autograd-loop burde opføre sig præcis som det håndregnede — men skridtene bliver
vildere og vildere, og `a` eksploderer. Én vigtig linje er faldet ud af loopet. Hvilken?
(Tænk tilbage på opgave 6.4...)

In [ ]:
laeringsrate = 0.1
a = torch.tensor(-2.0, requires_grad=True)
for i in range(15):
    tab = (a - 3) ** 2
    tab.backward()
    with torch.no_grad():
        a -= laeringsrate * a.grad
    a.grad.zero_()   # ← den manglende linje: nulstil, ellers hober gradienterne sig op
print("ender i a =", round(a.item(), 4))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Uden <code>a.grad.zero_()</code> lægges hver ny gradient oven i alle de gamle (opgave 6.4!), så skridtene vokser ukontrolleret. Symptomet — et tab der pludselig eksploderer — er noget, rigtige ML-folk også glemmer og fejlsøger. Nu ved I, hvad I skal kigge efter.</span>

##### Opgave 7.6 ⭐
Funktionen $f(x) = x^4 - 3x^2 + x$ har **to dale**. Kør gradient descent med start i
$x = 2$ og bagefter i $x = -2$ (læringsrate 0.01, 80 skridt). I lander to forskellige
steder! Hvad betyder det for gradient descent som metode — finder den altid *det bedste*
minimum?

In [ ]:
def g(x):
    return x ** 4 - 3 * x ** 2 + x

for start in [2.0, -2.0]:
    x = torch.tensor(start, requires_grad=True)
    skridt = [x.item()]
    for i in range(80):
        tab = g(x)
        tab.backward()
        with torch.no_grad():
            x -= 0.01 * x.grad
        x.grad.zero_()
        skridt.append(x.item())
    print(f"start = {start}: ender i x = {round(x.item(), 3)}, hvor g(x) = {round(g(torch.tensor(x.item())).item(), 3)}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Fra x = 2 lander man i den højre dal (x ≈ 1,13), fra x = −2 i den venstre (x ≈ −1,30) — og den venstre er faktisk DYBEST. Gradient descent ruller bare ned i den nærmeste dal (<b>et lokalt minimum</b>) og bliver der; den garanterer ikke at finde verdens bedste løsning. Neurale netværks tabslandskaber har MILLIONER af dale — i praksis viser de lokale minima sig heldigvis oftest at være gode nok.</span>

##### Opgave 7.7
Udfyld MSE-linjen i regressionsloopet (kvadrér fejlen og tag gennemsnittet — se formlen i
kommentaren).

In [ ]:
x = torch.tensor(df["Attack"].values, dtype=torch.float32)
y = torch.tensor(df["Total"].values, dtype=torch.float32)
x = (x - x.mean()) / x.std()
y = (y - y.mean()) / y.std()

a = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
for epoke in range(100):
    y_hat = a * x + b
    tab = ((y_hat - y) ** 2).mean()
    tab.backward()
    with torch.no_grad():
        a -= 0.1 * a.grad
        b -= 0.1 * b.grad
    a.grad.zero_()
    b.grad.zero_()
print("tab:", round(tab.item(), 4), "| linje: y =", round(a.item(), 3), "· x +", round(b.item(), 3))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>((y_hat - y) ** 2).mean()</code> — fejlen for hver af de 800 Pokémon kvadreres (så positive og negative fejl ikke ophæver hinanden), og gennemsnittet er tabet. Hældningen ender på ca. 0,74 — præcis korrelationen mellem Attack og Total fra notebook 1 (ikke et tilfælde, men det er en historie til en anden dag 😉).</span>

##### Opgave 7.8
1D-regressionen forudsagde `Total` ud fra `Attack`. Skift x-variablen til `Defense`, `HP`
og `Speed` (én ad gangen — genbrug cellen fra opgave 7.7) og notér sluttabet for hver.
Hvilken enkelt stat er den bedste "Total-spåkugle"?

In [ ]:
y = torch.tensor(df["Total"].values, dtype=torch.float32)
y = (y - y.mean()) / y.std()

for kolonne in ["Attack", "Defense", "HP", "Speed", "Sp. Atk"]:
    x_k = torch.tensor(df[kolonne].values, dtype=torch.float32)
    x_k = (x_k - x_k.mean()) / x_k.std()
    a = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    for epoke in range(100):
        y_hat = a * x_k + b
        tab = ((y_hat - y) ** 2).mean()
        tab.backward()
        with torch.no_grad():
            a -= 0.1 * a.grad
            b -= 0.1 * b.grad
        a.grad.zero_()
        b.grad.zero_()
    print(f"{kolonne:8s}: sluttab = {round(tab.item(), 3)}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Sp. Atk og Attack giver de laveste sluttab (bedste enkelt-forudsigelser), Speed det højeste — samme rangorden som korrelationerne med Total i notebook 1 (Sp. Atk 0,75 / Attack 0,74 / Speed 0,58). Lavt tab og høj korrelation er to sider af samme sag.</span>

##### Opgave 7.9 ⭐
Kør multi-feature-regressionen **uden** at dele med 100 — altså på de rå tal (fjern
`/ 100` begge steder). Hvad sker der med tabet? Prøv at redde situationen ved at gøre
læringsraten meget mindre (fx 0.00001). Hvad lærer det os om skalering af data?

In [ ]:
X_raa = torch.tensor(df[stats].values, dtype=torch.float32)
y_raa = torch.tensor(df["Total"].values, dtype=torch.float32)

for laeringsrate in [0.1, 0.00001]:
    w = torch.zeros(6, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    for epoke in range(50):
        y_hat = X_raa @ w + b
        tab = ((y_hat - y_raa) ** 2).mean()
        tab.backward()
        with torch.no_grad():
            w -= laeringsrate * w.grad
            b -= laeringsrate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    print(f"lr = {laeringsrate}: tab efter 50 epoker = {tab.item():.4g}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Med lr = 0.1 på rå tal eksploderer tabet til NaN på få epoker — tallene (og dermed gradienterne) er ~100× større, så skridtene bliver alt for lange. Med den håndplukkede mini-læringsrate 0.00001 overlever træningen (og kommer faktisk pænt ned her) — men prøv 0.001 eller 0.0001: stadig eksplosion eller hakkende kaos. Moralen: på uskalerede data bliver læringsraten et skrøbeligt lotteri, man skal finjustere pr. datasæt — <b>skalér/standardisér dine features</b>, så virker standardvalget bare. Det er præcis derfor, notebook 1 sluttede med standardisering.</span>

##### Opgave 7.10
Udfyld tre-trins-rytmen (`zero_grad` → `backward` → `step`) i den indbyggede version.

In [ ]:
X = torch.tensor(df[stats].values, dtype=torch.float32) / 100   # samme data som før
y = torch.tensor(df["Total"].values, dtype=torch.float32) / 100

model = nn.Linear(6, 1)
with torch.no_grad():
    model.weight.zero_()   # start i nul ligesom den manuelle version, så turen er den samme
    model.bias.zero_()
tabsfunktion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoke in range(3000):
    optimizer.zero_grad()
    y_hat = model(X).squeeze()
    tab = tabsfunktion(y_hat, y)
    tab.backward()
    optimizer.step()

print("vægte:", model.weight.detach().numpy().round(3))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>zero_grad()</code> (nulstil — opgave 6.4!), <code>backward()</code> (find gradienter), <code>step()</code> (tag skridtet). Denne rytme er skelettet i AL træning i PyTorch — fra denne lille model til ChatGPT. Lær den i søvne. (Detalje: vi nulstiller startvægtene, fordi nn.Linear ellers starter i TILFÆLDIGE vægte — og fra nogle startpunkter tager de sidste decimaler mange tusinde ekstra epoker i denne meget flade dal. Prøv selv uden nulstillingen!)</span>

##### Opgave 7.11
I regressionsemnet fandt I den bedste linje med en formel — på ét sekund, uden løkker.
Hvorfor gider vi så overhovedet bøvle med gradient descent? Og hvad var det egentlig,
gradient descent "opdagede" i multi-feature-eksemplet, som vi aldrig fortalte den?

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Formlen findes KUN for lineær regression (og få andre simple modeller). Neurale netværk er så komplicerede funktioner, at intet menneske kan udlede en løsningsformel — men gradient descent er ligeglad: den skal bare bruge gradienter, og dem leverer autograd uanset modellens kompleksitet. Og i multi-feature-eksemplet opdagede GD helt selv, at Total = summen af de seks stats (alle vægte → 1, bias → 0) — mønstret lå i dataene.</span>

# Godt gået! 🎉

I har nu hele motoren: tensorer, autograd og gradient descent — og I har set PyTorch finde
mønstre i data helt selv.

**Næste stop:** `3-Neurale-netvaerk.ipynb`, hvor `nn.Linear`-klodserne stables til et rigtigt
neuralt netværk (med jeres klasse-viden fra intro-programmering i hovedrollen), og hvor I
træner det til at spotte legendariske Pokémon. 🌟